In [1]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
import sys
import os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults

C:\Users\davis\AppData\Local\Temp\ipykernel_13748\723937994.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [3]:
model_client = OpenAIChatCompletionClient(
    model='gpt-3.5-turbo'
)

In [4]:
async def research_tool(query: str) -> str:
    """Search the web for information"""
    search = TavilySearchResults(max_results=3)
    results = search.invoke(query)
    return str(results)

In [5]:
supervisor_agent = AssistantAgent(
    "SupervisorAgent",
    description="An agent for planning and supervising which agent to act next",
    model_client=model_client,
    system_message="""You are a supervisor AI. Decide which agent should act next
            Options:
            - research_agent → if more information is needed.
            - writer_agent → if a summary or refinement is needed.
            - end → if the task is complete
            Respond ONLY with one of the following: research_agent, writer_agent, or end."""
)

In [6]:
research_agent = AssistantAgent(
    "ResearchAgent",
    description="An agent for researching about the given topic using the tool provided",
    model_client=model_client,
    tools= [research_tool],
    system_message="""
    You are a researcher agent, and once you are given a topic to research about, you should make use of tool provided to you for the research purpose.
    You should only be researching based on the tool provided to you
    """,
)

In [7]:
writer_agent = AssistantAgent(
    "WriterAgent",
    description="An agent for summarizing the research",
    model_client=model_client,
    system_message="""
    You will be given the research results, and your task is only the summarize them in a professional manner within 150 words
    """,
)

In [10]:
text_mention_termination = TextMentionTermination("end")
max_messages_termination = MaxMessageTermination(max_messages=5)
termination = text_mention_termination | max_messages_termination

In [11]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [12]:
team = SelectorGroupChat(
    [supervisor_agent, writer_agent, research_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  # Allow an agent to speak multiple turns in a row.
)


In [13]:
task = "What are implications of Generative AI in the field of cricket (sports)"

In [14]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
What are implications of Generative AI in the field of cricket (sports)
---------- TextMessage (SupervisorAgent) ----------
research_agent
---------- TextMessage (SupervisorAgent) ----------
Research_agent
---------- ToolCallRequestEvent (ResearchAgent) ----------
[FunctionCall(id='call_ka08UPnggLnZY5LU2BoLmXTN', arguments='{"query":"Implications of Generative AI in cricket sports"}', name='research_tool')]
---------- ToolCallExecutionEvent (ResearchAgent) ----------
[FunctionExecutionResult(content='1 validation error for TavilySearchAPIWrapper\n  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/value_error', name='research_tool', call_id='call_ka08UPnggLnZY5LU2BoLmXTN', is_error=True)]
---------- ToolCallSummaryMes

C:\Users\davis\AppData\Local\Temp\ipykernel_13748\309514807.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=3)


---------- ThoughtEvent (ResearchAgent) ----------
It seems there is an issue with the research tool. Let me retry the search request with the correct tool.
---------- ToolCallRequestEvent (ResearchAgent) ----------
[FunctionCall(id='call_DLlNMVMMMy6h60dxX7AsIkeB', arguments='{"query": "Implications of Generative AI in cricket sports"}', name='research_tool')]
---------- ToolCallExecutionEvent (ResearchAgent) ----------
[FunctionExecutionResult(content='1 validation error for TavilySearchAPIWrapper\n  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/value_error', name='research_tool', call_id='call_DLlNMVMMMy6h60dxX7AsIkeB', is_error=True)]
---------- ToolCallSummaryMessage (ResearchAgent) ----------
1 validation error for TavilySearchAPIWrapper
  Value err

TaskResult(messages=[TextMessage(id='ae42505c-ff0f-46bd-93a8-b84eeb63758c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 2, 18, 23, 42, 652829, tzinfo=datetime.timezone.utc), content='What are implications of Generative AI in the field of cricket (sports)', type='TextMessage'), TextMessage(id='3d78b2fc-db19-44ca-ba1e-6def3893e9e8', source='SupervisorAgent', models_usage=RequestUsage(prompt_tokens=95, completion_tokens=2), metadata={}, created_at=datetime.datetime(2026, 6, 2, 18, 23, 46, 256893, tzinfo=datetime.timezone.utc), content='research_agent', type='TextMessage'), TextMessage(id='19c88466-bc8a-47ff-9cdd-39f8a93f49a4', source='SupervisorAgent', models_usage=RequestUsage(prompt_tokens=101, completion_tokens=2), metadata={}, created_at=datetime.datetime(2026, 6, 2, 18, 23, 48, 336510, tzinfo=datetime.timezone.utc), content='Research_agent', type='TextMessage'), ToolCallRequestEvent(id='1f0f493e-c22f-41e1-8b7d-19d2e5dce070', source='ResearchAg